# Virchow Training on WILDS CAMELYON17 (Colab, Beginner)

Trains `scripts/train_virchow_preprocessed_colab.py` on **preprocessed WILDS H5** (`train_x/y.h5`, `valid_x/y.h5`).

This is the WILDS version of your previous Colab training flow:
- checkpoints/results saved to Google Drive (`RUN_DIR`)
- optional copy of preprocessed H5 to local SSD (`/content`) for faster reads
- resume-safe training (`--resume`)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!nvidia-smi


In [ ]:
%cd /content
!git clone https://github.com/<your-username>/<your-repo>.git GP_ECG
%cd /content/GP_ECG


If you already uploaded your repo into Drive instead of cloning, skip the previous cell and set `REPO_DIR` below to that Drive path.


In [ ]:
# ==== EDIT THESE 4 LINES ====
REPO_DIR = '/content/GP_ECG'
PREPROCESSED_DIR = '/content/drive/MyDrive/GP_ECG_DATA/wilds/camelyon17_h5_full_oodval/preprocessed_macenko_benchmark_style'
RUN_DIR = '/content/drive/MyDrive/GP_ECG_RUNS/virchow_wilds_preprocessed_run_01'
HF_TOKEN = ''  # paste token if Virchow access is gated

COPY_PREPROCESSED_TO_LOCAL_SSD = True

import os
os.makedirs(RUN_DIR, exist_ok=True)
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOKEN

print('REPO_DIR:', REPO_DIR)
print('PREPROCESSED_DIR:', PREPROCESSED_DIR)
print('RUN_DIR:', RUN_DIR)


In [ ]:
import os
import shutil

LOCAL_PREPROCESSED_DIR = '/content/local_wilds_preprocessed_h5'

if COPY_PREPROCESSED_TO_LOCAL_SSD:
    if not os.path.isdir(PREPROCESSED_DIR):
        raise FileNotFoundError('Drive path not found: ' + PREPROCESSED_DIR)
    if os.path.exists(LOCAL_PREPROCESSED_DIR):
        shutil.rmtree(LOCAL_PREPROCESSED_DIR)
    print('Copying preprocessed WILDS H5 folder Drive -> local SSD ...')
    shutil.copytree(PREPROCESSED_DIR, LOCAL_PREPROCESSED_DIR)
    PREPROCESSED_DIR = LOCAL_PREPROCESSED_DIR
    print('Done. Using local:', PREPROCESSED_DIR)
else:
    print('Using Drive path directly:', PREPROCESSED_DIR)


In [ ]:
%cd {REPO_DIR}
!python -m pip install --upgrade pip
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install timm h5py tqdm huggingface_hub scikit-learn


Optional if model access is gated: run login once and paste your Hugging Face token.


In [ ]:
# !huggingface-cli login


In [ ]:
import os
required = [
    os.path.join(PREPROCESSED_DIR, 'train_x.h5'),
    os.path.join(PREPROCESSED_DIR, 'train_y.h5'),
    os.path.join(PREPROCESSED_DIR, 'valid_x.h5'),
    os.path.join(PREPROCESSED_DIR, 'valid_y.h5'),
]
for p in required:
    print(('OK ' if os.path.exists(p) else 'MISSING '), p)


Run training. Notes:
- `--resume` lets you restart after disconnect without losing progress.
- `--save-every-epoch-copy` keeps extra checkpoints (takes more Drive space).
- If `valid_*` was built from `--valid-source val`, this is OOD validation.


In [ ]:
%cd {REPO_DIR}
!python scripts/train_virchow_preprocessed_colab.py \
  --preprocessed-dir "{PREPROCESSED_DIR}" \
  --out-dir "{RUN_DIR}" \
  --epochs 10 \
  --batch-size 64 \
  --num-workers 2 \
  --head-dropout 0.2 \
  --mc-samples 30 \
  --resume \
  --save-every-epoch-copy


In [ ]:
import json, os
print('Run dir:', RUN_DIR)
artifacts = [
    'checkpoint_last.pt', 'model_best.pt', 'metrics_history.json', 'metrics_final.json',
    'metrics_final_detailed.json', 'temperature_fit.json', 'run_config.json', 'run_manifest.json',
    'run_progress.json', 'val_predictions.npz'
]
for name in artifacts:
    p = os.path.join(RUN_DIR, name)
    print(('OK ' if os.path.exists(p) else 'MISSING '), p)

compact = os.path.join(RUN_DIR, 'metrics_final.json')
if os.path.exists(compact):
    with open(compact, 'r', encoding='utf-8') as f:
        print('\nmetrics_final.json:')
        print(json.dumps(json.load(f), indent=2))
